# Phase 0 — Environment setup (bio-unlearning tamper-resistance study)

Run this notebook in Google Colab with a GPU runtime.

**Before running:** `Runtime` -> `Change runtime type` -> Hardware accelerator = `T4 GPU` (or any available GPU) -> Save.

Run the cells top to bottom, then copy all the printed output back to Claude Code so it can be recorded in the project's `results/` and `requirements.txt`.

Reminder (project safety rule): this notebook only loads a base instruction-tuned model and runs a benign test prompt. No hazardous content is generated, stored, or trained on here.

In [ ]:
!nvidia-smi

In [ ]:
# Install pinned versions of everything except torch (Colab already ships a CUDA-enabled torch build; reinstalling it is slow and can break the Colab runtime).
!pip install -q transformers==5.14.1 datasets==5.0.0 accelerate==1.14.0 peft==0.19.1 lm-eval==0.4.12

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL} ...")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Params (B): {sum(p.numel() for p in model.parameters())/1e9:.2f}")
print(f"num_hidden_layers: {model.config.num_hidden_layers}")
print(f"hidden_size: {model.config.hidden_size}")

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is the capital of France?"}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt").to(model.device)

t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print(f"Generated in {time.time()-t0:.1f}s")

text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("---GENERATION---")
print(text)

In [ ]:
# Capture the exact working environment for requirements.txt / reproducibility.
!pip freeze